In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_sales_catalog.silver;
CREATE SCHEMA IF NOT EXISTS retail_sales_catalog.quarantine;

In [0]:
CATALOG = 'retail_sales_catalog'

def read_bronze(table_name):
    return spark.table(f"{CATALOG}.bronze.{table_name}")


def write_silver_and_quarantine(df, table_name, primary_keys, validation_column="_validation_error"):
    """
    Splits validated data into Silver and Quarantine tables. 

    Valid records: validation_column IS NULL

    Invalid records: validation_column IS NOT NULL
    """

    # Separate the valid and invalid records according to normal validation rule 
    valid_candidates = df.filter(col(validation_column).isNull())

    invalid_df = df.filter(col(validation_column).isNotNull())

    #Detect duplicate records
    window_spec = Window.partitionBy(*primary_keys).orderBy(col("_ingested_at").desc())

    ranked_df = valid_candidates.withColumn("_row_number", row_number().over(window_spec))

    valid_df = ranked_df.filter(col("_row_number")==1).drop("_row_number", validation_column)

    # Quarantine older duplicate records
    duplicate_df = ranked_df.filter(col("_row_number")>1)\
        .drop("_row_number")\
            .withColumn(validation_column, lit('DUPLICATE PRIMARY KEY'))

    # Combine all rejected records
    quarantine_df = invalid_df.unionByName(duplicate_df, allowMissingColumns = True)

    # Write valid records to Silver
    valid_df.write.format('delta')\
        .mode('overwrite')\
            .option('overwriteSchema', 'true')\
                .saveAsTable(f"{CATALOG}.silver.{table_name}")

    #Write invalid records to Quarantine
    quarantine_df.write.format('delta')\
        .mode('overwrite')\
            .option('overwriteSchema', 'true')\
                .saveAsTable(f"{CATALOG}.quarantine.{table_name}")

    return valid_df, invalid_df


def validate_foreign_key(df, foreign_key, parent_table, parent_key, error_message):
    parent_df = spark.table(parent_table).select(col(parent_key).alias('_parent_key'))
    result = df.join(parent_df, df[foreign_key] == parent_df['_parent_key'], "left")\
        .withColumn('_validation_error', 
                    when(col('_validation_error').isNotNull(), col('_validation_error'))
                    .when(col('_parent_key').isNull(), lit(error_message)))\
                    .drop('_parent_key')

    return result

Customers Table

In [0]:
customers_df = read_bronze('customers')
customers_df.printSchema()
display(customers_df.limit(10))

In [0]:
#Checking null 
customers_df.filter(col('customer_id').isNull()).count()

#Checking duplicates
customers_df.groupBy('customer_id').count().filter(col('count')>1).show()

In [0]:
#Creating the silver transformation
silver_customers = (
    customers_df
    .withColumn("customer_id",col('customer_id').cast('int'))
    .withColumn("signup_date", to_date(col('signup_date')))
    .withColumn("customer_name", trim(col('customer_name')))
    .withColumn("city", initcap(trim(col('city'))))
    .withColumn('_processed_at', current_timestamp())
    .withColumn('_validation_error', 
                when(col('customer_id').isNull(), lit('MISSING_CUSTOMER_ID'))
                .when(col('customer_name').isNull(), lit('MISSING_CUSTOMER_NAME'))
                .when(col('signup_date').isNull(), lit('MISSING_SIGNUP_DATE')))
)

silver_customers, quarantine_customers = write_silver_and_quarantine(silver_customers, table_name='customers', primary_keys=['customer_id'])


In [0]:
#validating silver against bronze

bronze_count = customers_df.count()
silver_count = silver_customers.count()
quarantine_count = quarantine_customers.count()

print(f"Bronze: {bronze_count}")
print(f"Silver: {silver_count}")
print(f"Quarantine: {quarantine_count}")

In [0]:
%sql
SELECT * FROM retail_sales_catalog.silver.customers LIMIT 5;

Categories Table

In [0]:
categories_df = read_bronze('categories')
categories_df.printSchema()
display(categories_df.limit(10))

In [0]:
#Checking null 
categories_df.filter(col('category_id').isNull()).count()

In [0]:
#Checking duplicates
categories_df.groupBy('category_id').count().filter(col('count')>1).show()

In [0]:
#Creating the silver transformation
transformed_categories = (
    categories_df
    .withColumn('category_id', col('category_id').cast('int'))
    .withColumn('category_name', trim(col('category_name')))
    .withColumn('_processed_at', current_timestamp())
    .withColumn('_validation_error',
                when(col('category_id').isNull(), lit('MISSING_CATEGORY_ID'))
                .when(col('category_name').isNull(), lit('MISSING_CATEGORY_NAME')))
)

silver_categories, quarantine_categories = write_silver_and_quarantine(transformed_categories, table_name ='categories', primary_keys=['category_id'])


In [0]:
%sql
SELECT * FROM retail_sales_catalog.silver.categories LIMIT 5

Stores

In [0]:
stores_df = read_bronze('stores')
stores_df.printSchema()
display(stores_df.limit(10))

In [0]:
stores_df.filter(col('store_id').isNull()).count()

In [0]:
stores_df.groupBy('store_id').count().filter(col('count')>1).show()

In [0]:
transformed_stores = (stores_df
                      .withColumn('store_id', col('store_id').cast('int'))
                      .withColumn('store_name', trim(col('store_name')))
                      .withColumn('city', initcap(trim(col('city'))))
                      .withColumn('_processed_at', current_timestamp())
                      .withColumn('_validation_error',
                                  when(col('store_id').isNull(), lit('MISSING_STORE_ID'))
                                  .when(col('store_name').isNull(), lit('MISSING_STORE_NAME'))
                                  .when(col('city').isNull(), lit('MISSING_CITY')))
                      )

silver_store, quarantine_store = write_silver_and_quarantine(transformed_stores, table_name ='stores', primary_keys=['store_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.silver.stores LIMIT 5

Promotions

In [0]:
promotions_df = read_bronze('promotions')
promotions_df.printSchema()
display(promotions_df.limit(10))

In [0]:
promotions_df.filter(col('promotion_id').isNull()).count()

In [0]:
promotions_df.groupBy('promotion_id').count().filter(col('count')>1).show()

In [0]:
transformed_promotion = (promotions_df
                         .withColumn('promotion_id', col('promotion_id').cast('int'))
                         .withColumn('discount', col('discount').cast('int'))
                         .withColumn('_processed_at', current_timestamp())
                         .withColumn('_validation_error',
                                     when(col('promotion_id').isNull(), lit('MISSING_PROMOTION_ID'))
                                     .when(col('discount').isNull(), lit('MISSING_DISCOUNT_AMOUNT'))
                                     .when(col('discount')<0, lit('NEGATIVE_DISCOUNT_AMOUNT'))))

silver_promotion, quarantine_promotion = write_silver_and_quarantine(transformed_promotion, table_name='promotions', primary_keys=['promotion_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.promotions LIMIT 5

Suppliers

In [0]:
supplier_df = read_bronze('suppliers')
supplier_df.printSchema()
display(supplier_df.limit(10))

In [0]:
supplier_df.filter(col('supplier_id').isNull()).count()

In [0]:
supplier_df.groupBy('supplier_id').count().filter(col('count')>1).show()

In [0]:
transformed_supplier = (supplier_df
                        .withColumn('supplier_id', col('supplier_id').cast('int'))
                        .withColumn('supplier_name', trim(col('supplier_name')))
                        .withColumn('country', initcap(trim(col('country'))))
                        .withColumn('_processed_at', current_timestamp())
                        .withColumn('_validation_error',
                                    when(col('supplier_id').isNull(), lit('MISSING_SUPPLIER_ID'))
                                    .when(col('supplier_name').isNull(), lit('MISSING_SUPPLIER_NAME'))
                                    .when(col('country').isNull(), lit('MISSING_SUPPLIER_COUNTRY'))))

silver_supplier, quarantine_supplier = write_silver_and_quarantine(transformed_supplier, table_name='suppliers', primary_keys=['supplier_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.silver.suppliers 

Employees

In [0]:
employee_df = read_bronze('employees')
employee_df.printSchema()
display(employee_df.limit(10))

In [0]:
employee_df.filter(col('employee_id').isNull()).count()

In [0]:
employee_df.groupBy('employee_id').count().filter(col('count')>1).show()

In [0]:
transformed_employee = (employee_df
                        .withColumn('employee_id', col('employee_id').cast('int'))
                        .withColumn('store_id', col('store_id').cast('int'))
                        .withColumn('employee_name', trim('employee_name'))
                        .withColumn('salary', col('salary').cast('int'))
                        .withColumn('_processed_at', current_timestamp())
                        .withColumn('_validation_error',
                                    when(col('employee_id').isNull(), lit('MISSING_EMPLOYEE_ID'))
                                    .when(col('store_id').isNull(), lit('MISSING_STORE_ID'))
                                    .when(col('employee_name').isNull(), lit('MISSING_EMPLOYEE_NAME'))
                                    .when(col('salary').isNull(), lit('MISSING_SALARY'))
                                    .when(col('salary')<0, lit('INVALID_SALARY'))))

transformed_employee = validate_foreign_key(transformed_employee,
                                            foreign_key = 'store_id',
                                            parent_table = 'retail_sales_catalog.silver.stores',
                                            parent_key = 'store_id',
                                            error_message = 'INVALID_STORE_ID')

silver_employee, quarantine_employee = write_silver_and_quarantine(transformed_employee, table_name='employees', primary_keys=['employee_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.employees

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.stores

Products

In [0]:
products_df = read_bronze('products')
products_df.printSchema()
display(products_df.limit(10))

In [0]:
products_df.filter(col('product_id').isNull()).count()

In [0]:
products_df.groupBy('product_id').count().filter(col('count')>1).show()

In [0]:
transformed_products = (products_df
                        .withColumn('product_id', col('product_id').cast('int'))
                        .withColumn('category_id', col('category_id').cast('int'))
                        .withColumn('supplier_id', col('supplier_id').cast('int'))
                        .withColumn('product_name', trim(col('product_name')))
                        .withColumn('price', col('price').cast('int'))
                        .withColumn('_processed_at', current_timestamp())
                        .withColumn('_validation_error',
                                    when(col('product_id').isNull(), lit('MISSING_PRODUCT_ID'))
                                    .when(col('category_id').isNull(), lit('MISSING_CATEGORY_ID'))
                                    .when(col('supplier_id').isNull(), lit('MISSING_SUPPLIER_ID'))
                                    .when(col('product_name').isNull(), lit('MISSING_PRODUCT_NAME'))
                                    .when(col('price').isNull(), lit('MISSING_PRODUCT_PRICE'))
                                    .when(col('price')<0, lit('INVALID_PRODUCT_PRICE'))))

transformed_products = validate_foreign_key(transformed_products,
                                            foreign_key='category_id',
                                            parent_table='retail_sales_catalog.silver.categories',
                                            parent_key = 'category_id',
                                            error_message= 'INVALID_CATEGORY_ID')

transformed_products = validate_foreign_key(transformed_products,
                                            foreign_key = 'supplier_id',
                                            parent_table = 'retail_sales_catalog.silver.suppliers',
                                            parent_key = 'supplier_id',
                                            error_message = 'INVALID_SUPPLIER_ID')

silver_products, quarantine_products = write_silver_and_quarantine(transformed_products,
                                                                   table_name='products',
                                                                   primary_keys = ['product_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.products

Orders

In [0]:
orders_df = read_bronze('orders')
orders_df.printSchema()
display(orders_df.limit(10))

In [0]:
orders_df.filter(col('order_id').isNull()).count()

In [0]:
orders_df.groupBy('order_id').count().filter(col('count')>1).show()

In [0]:
transformed_orders = (orders_df
                      .withColumn('order_id', col('order_id').cast('int'))
                      .withColumn('customer_id', col('customer_id').cast('int'))
                      .withColumn('store_id', col('store_id').cast('int'))
                      .withColumn('promotion_id', col('promotion_id').cast('int'))
                      .withColumn('order_date', col('order_date').cast('date'))
                      .withColumn('_processed_at', current_timestamp())
                      .withColumn('_validation_error',
                                  when(col('order_id').isNull(), lit('MISSING_ORDER_ID'))
                                  .when(col('customer_id').isNull(), lit('MISSING_CUSTOMER_ID'))
                                  .when(col('store_id').isNull(), lit('MISSING_STORE_ID'))
                                  .when(col('promotion_id').isNull(), lit('MISSING_PROMOTION_ID'))
                                  .when(col('order_date').isNull(), lit('MISSING_ORDER_DATE'))
                                  .when(col('order_date')>current_date(), lit('INVALID_ORDER_DATE'))))

transformed_orders = validate_foreign_key(transformed_orders,
                                          foreign_key = 'customer_id',
                                          parent_table = 'retail_sales_catalog.silver.customers',
                                          parent_key='customer_id',
                                          error_message= 'INVALID_CUSTOMER_ID')

transformed_orders = validate_foreign_key(transformed_orders,
                                           foreign_key = 'store_id',
                                           parent_table = 'retail_sales_catalog.silver.stores',
                                           parent_key = 'store_id',
                                           error_message='INVALID_STORE_ID')

transformed_orders = validate_foreign_key(transformed_orders,
                                          foreign_key = 'promotion_id',
                                          parent_table = 'retail_sales_catalog.silver.promotions',
                                          parent_key = 'promotion_id',
                                          error_message= 'INVALID_PROMOTION_ID')

silver_orders, quarantine_orders = write_silver_and_quarantine(transformed_orders,
                                                               table_name='orders',
                                                               primary_keys=['order_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.orders

Order Items

In [0]:
order_items_df = read_bronze('order_items')
order_items_df.printSchema()
display(order_items_df.limit(10))

In [0]:
order_items_df.filter(col('order_item_id').isNull()).count()

In [0]:
order_items_df.groupBy('order_item_id').count().filter(col('count')>1).show()

In [0]:
transformed_order_items = (order_items_df
                           .withColumn('order_item_id', col('order_item_id').cast('int'))
                           .withColumn('order_id', col('order_id').cast('int'))
                           .withColumn('product_id', col('product_id').cast('int'))
                           .withColumn('qty', col('qty').cast('int'))
                           .withColumn('price', col('price').cast('int'))
                           .withColumn('_processed_at', current_timestamp())
                           .withColumn('_validation_error',
                                       when(col('order_item_id').isNull(), lit('MISSING_ORDER_ITEM_ID'))
                                       .when(col('order_id').isNull(), lit('MISSING_ORDER_ID'))
                                       .when(col('product_id').isNull(), lit('MISSING_PRODUCT_ID'))
                                       .when(col('qty').isNull(), lit('MISSING_QUANTITY'))
                                       .when(col('qty')<0, lit('INVALID_QUANTITY'))
                                       .when(col('price').isNull(), lit('MISSING_PRICE'))
                                       .when(col('price')<0, lit('INVALID_PRICE'))))

transformed_order_items = validate_foreign_key(transformed_order_items,
                                               foreign_key= 'order_id',
                                               parent_table='retail_sales_catalog.silver.orders',
                                               parent_key='order_id',
                                               error_message='INVALID_ORDER_ID')

transformed_order_items = validate_foreign_key(transformed_order_items,
                                               foreign_key='product_id',
                                               parent_table = 'retail_sales_catalog.silver.products',
                                               parent_key = 'product_id',
                                               error_message = 'INVALID_PRODUCT_ID')

silver_order_items, quarantine_order_items = write_silver_and_quarantine(transformed_order_items,
                                                                         table_name='order_items',
                                                                         primary_keys=['order_item_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.silver.order_items

Payments

In [0]:
payments_df = read_bronze('payments')
payments_df.printSchema()
display(payments_df.limit(10))

In [0]:
payments_df.filter(col('payment_id').isNull()).count()

In [0]:
payments_df.groupBy('payment_id').count().filter(col('count') > 1).show()

In [0]:
transformed_payments = (payments_df
                        .withColumn('payment_id', col('payment_id').cast('int'))
                        .withColumn('order_id', col('order_id').cast('int'))
                        .withColumn('amount', col('amount').cast('int'))
                        .withColumn('_processed_at', current_timestamp())
                        .withColumn('_validation_error',
                                    when(col('payment_id').isNull(), lit('MISSING_PAYMENT_ID'))
                                    .when(col('order_id').isNull(), lit('MISSING_ORDER_ID'))
                                    .when(col('amount').isNull(), lit('MISSING_PAYMENT_AMOUNT'))
                                    .when(col('amount')<0, lit('INVALID_PAYMENT_AMOUNT'))))

transformed_payments = validate_foreign_key(transformed_payments,
                                            foreign_key='order_id',
                                            parent_table = 'retail_sales_catalog.silver.orders',
                                            parent_key='order_id',
                                            error_message='INVALID_ORDER_ID')

silver_payments, quarantine_payments = write_silver_and_quarantine(transformed_payments,
                                                                   table_name='payments',
                                                                   primary_keys=['payment_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.payments

Returns

In [0]:
returns_df = read_bronze('returns')
returns_df.printSchema()
display(returns_df.limit(10))

In [0]:
returns_df.filter(col('return_id').isNull()).count()

In [0]:
returns_df.groupBy('return_id').count().filter(col('count')>1).show()

In [0]:
transformed_returns = (returns_df
                       .withColumn('return_id', col('return_id').cast('int'))
                       .withColumn('order_item_id', col('order_item_id').cast('int'))
                       .withColumn('refund', col('refund').cast('int'))
                       .withColumn('_processed_at', current_timestamp())
                       .withColumn('_validation_error',
                                   when(col('return_id').isNull(), lit('MISSING_RETURN_ID'))
                                   .when(col('order_item_id').isNull(), lit('MISSING_ORDER_ITEM_ID'))
                                   .when(col('refund').isNull(), lit('MISSING_REFUND_VALUE'))
                                   .when(col('refund')<0, lit('INVALID_REFUND_VALUE'))))


transformed_returns = validate_foreign_key(transformed_returns,
                                           foreign_key = 'order_item_id',
                                           parent_table = 'retail_sales_catalog.silver.order_items',
                                           parent_key = 'order_item_id',
                                           error_message='INVALID_ORDER_ITEM_ID')

silver_returns, quarantine_returns = write_silver_and_quarantine(transformed_returns,
                                                                 table_name = 'returns',
                                                                 primary_keys = ['return_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.returns

Shipments

In [0]:
shipments_df = read_bronze('shipments')
shipments_df.printSchema()
display(shipments_df.limit(10))

In [0]:
shipments_df.filter(col('shipment_id').isNull()).count()

In [0]:
shipments_df.groupBy('shipment_id').count().filter(col('count')>1).show()

In [0]:
transformed_shipments = (shipments_df
                         .withColumn('shipment_id', col('shipment_id').cast('int'))
                         .withColumn('order_id', col('order_id').cast('int'))
                         .withColumn('status', trim(col('status')))
                         .withColumn('_processed_at', current_timestamp())
                         .withColumn('_validation_error',
                                     when(col('shipment_id').isNull(), lit('MISSING_SHIPMENT_ID'))
                                     .when(col('order_id').isNull(), lit('MISSING_ORDER_ID'))
                                     .when(col('status').isNull(), lit('MISSING_SHIPMENT_STATUS'))
                                     .when(~col('status').isin(['delivered', 'shipped', 'late']), lit('INVALID_SHIPMENT_STATUS'))))

transformed_shipments = validate_foreign_key(transformed_shipments,
                                             foreign_key = 'order_id',
                                             parent_table = 'retail_sales_catalog.silver.orders',
                                             parent_key ='order_id',
                                             error_message='INVALID_ORDER_ID')

silver_shipments, quarantine_shipments = write_silver_and_quarantine(transformed_shipments,
                                                                     table_name ='shipments',
                                                                     primary_keys =['shipment_id'])

In [0]:
%sql
SELECT * FROM retail_sales_catalog.quarantine.shipments

In [0]:
%sql
SELECT DISTINCT status FROM retail_sales_catalog.bronze.shipments